In [1]:

import os
import unittest
from dataclasses import dataclass

# ==========================================
# WEEK 5: TOKEN ECONOMICS
# Cost and Rate-Limit Budgeting
# ==========================================

# 1. Feature Budget Configuration

@dataclass
class FeatureBudget:
    requests_per_day: int
    avg_input_tokens: int
    avg_output_tokens: int
    peak_requests_per_minute: int
    input_price_per_million: float
    output_price_per_million: float
    rpm_limit: int
    tpm_limit: int
    monthly_budget_usd: float
    daily_budget_usd: float
    safety_margin_percent: float

    # 2. Calculate token usage, cost and capacity
    def calculate(self):

        if min(self.requests_per_day,
               self.avg_input_tokens,
               self.avg_output_tokens,
               self.peak_requests_per_minute) < 0:
            raise ValueError("Usage assumptions cannot be negative.")

        if self.safety_margin_percent < 0:
            raise ValueError("Safety margin cannot be negative.")

        input_day = self.requests_per_day * self.avg_input_tokens
        output_day = self.requests_per_day * self.avg_output_tokens

        daily_cost = (
            input_day * self.input_price_per_million
            + output_day * self.output_price_per_million
        ) / 1_000_000

        monthly_cost = daily_cost * 30

        margin = 1 + self.safety_margin_percent / 100

        peak_rpm = self.peak_requests_per_minute * margin

        peak_tpm = (
            self.peak_requests_per_minute
            * (self.avg_input_tokens + self.avg_output_tokens)
            * margin
        )

        return {
            "input_tokens_day": input_day,
            "output_tokens_day": output_day,
            "total_tokens_day": input_day + output_day,
            "daily_cost_usd": daily_cost,
            "monthly_cost_usd": monthly_cost,
            "peak_rpm_with_margin": peak_rpm,
            "peak_tpm_with_margin": peak_tpm,
            "rpm_ok": peak_rpm <= self.rpm_limit,
            "tpm_ok": peak_tpm <= self.tpm_limit,
            "daily_budget_ok": daily_cost * margin <= self.daily_budget_usd,
            "monthly_budget_ok": monthly_cost * margin <= self.monthly_budget_usd
        }


# 3. Sample assumptions from the lab manual

def main():

    budget = FeatureBudget(
        requests_per_day=2000,
        avg_input_tokens=1200,
        avg_output_tokens=300,
        peak_requests_per_minute=30,
        input_price_per_million=0.30,
        output_price_per_million=1.20,
        rpm_limit=60,
        tpm_limit=100000,
        monthly_budget_usd=50,
        daily_budget_usd=2,
        safety_margin_percent=20
    )

    result = budget.calculate()

    # 4. Print Token Economics Report

    print("\nTOKEN ECONOMICS REPORT")
    print("-" * 40)

    print(f"Input tokens/day: {result['input_tokens_day']:,}")
    print(f"Output tokens/day: {result['output_tokens_day']:,}")
    print(f"Total tokens/day: {result['total_tokens_day']:,}")

    print(f"Estimated daily cost: ${result['daily_cost_usd']:.4f}")
    print(f"Estimated monthly cost: ${result['monthly_cost_usd']:.2f}")

    print(f"Peak RPM with margin: {result['peak_rpm_with_margin']:.1f}")
    print(f"Peak TPM with margin: {result['peak_tpm_with_margin']:.0f}")

    # 5. Budget and capacity checks

    checks = {
        "RPM capacity": result["rpm_ok"],
        "TPM capacity": result["tpm_ok"],
        "Daily budget": result["daily_budget_ok"],
        "Monthly budget": result["monthly_budget_ok"]
    }

    print("\nCHECKS")
    print("-" * 40)

    for name, passed in checks.items():
        print(f"{name}: {'PASS' if passed else 'REVIEW'}")

    if all(checks.values()):
        print("\nRecommendation: WITHIN configured planning limits.")
    else:
        print("\nRecommendation: REVISE assumptions, limits, or design before build.")

    # 6. Automated tests

    class TestFeatureBudget(unittest.TestCase):

        def make_budget(self, **overrides):
            values = dict(
                requests_per_day=1000,
                avg_input_tokens=1000,
                avg_output_tokens=500,
                peak_requests_per_minute=10,
                input_price_per_million=1.0,
                output_price_per_million=2.0,
                rpm_limit=100,
                tpm_limit=50000,
                monthly_budget_usd=100,
                daily_budget_usd=5,
                safety_margin_percent=20
            )
            values.update(overrides)
            return FeatureBudget(**values)

        def test_cost_estimate(self):
            result = self.make_budget().calculate()
            self.assertAlmostEqual(result["daily_cost_usd"], 2.0)
            self.assertAlmostEqual(result["monthly_cost_usd"], 60.0)

        def test_capacity_passes(self):
            result = self.make_budget().calculate()
            self.assertTrue(result["rpm_ok"])
            self.assertTrue(result["tpm_ok"])

        def test_rpm_limit_fails(self):
            result = self.make_budget(rpm_limit=5).calculate()
            self.assertFalse(result["rpm_ok"])

        def test_monthly_budget_fails(self):
            result = self.make_budget(monthly_budget_usd=20).calculate()
            self.assertFalse(result["monthly_budget_ok"])

    print("\nAUTOMATED TEST RESULTS")
    print("-" * 40)

    suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestFeatureBudget)
    unittest.TextTestRunner(verbosity=2).run(suite)


if __name__ == "__main__":
    main()


test_capacity_passes (__main__.main.<locals>.TestFeatureBudget.test_capacity_passes) ... ok
test_cost_estimate (__main__.main.<locals>.TestFeatureBudget.test_cost_estimate) ... ok
test_monthly_budget_fails (__main__.main.<locals>.TestFeatureBudget.test_monthly_budget_fails) ... ok
test_rpm_limit_fails (__main__.main.<locals>.TestFeatureBudget.test_rpm_limit_fails) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.008s

OK



TOKEN ECONOMICS REPORT
----------------------------------------
Input tokens/day: 2,400,000
Output tokens/day: 600,000
Total tokens/day: 3,000,000
Estimated daily cost: $1.4400
Estimated monthly cost: $43.20
Peak RPM with margin: 36.0
Peak TPM with margin: 54000

CHECKS
----------------------------------------
RPM capacity: PASS
TPM capacity: PASS
Daily budget: PASS
Monthly budget: REVIEW

Recommendation: REVISE assumptions, limits, or design before build.

AUTOMATED TEST RESULTS
----------------------------------------
